# Unit 6 — PandaReachDense: Actor-Critic (A2C) for a robot arm

**Pass criterion:** `mean_reward − std_reward ≥ −3.5`

We teach a robot arm to move its end-effector to a **target point** that is randomly placed every episode.

- **Actor:** decides how to move the joints.
- **Critic:** after every step it judges "was this better or worse than expected?" → **advantage** = `r + γ·V(s') − V(s)`.

**How to run:** `Runtime → Run all` (~25–40 min incl. ~5 min setup, no GPU). The Hugging Face login reads the `HF_TOKEN` Colab secret.

## 1. Setup
`panda-gym` uses **PyBullet** as its physics engine; installation can take a few minutes.

In [ ]:
!pip install -q "stable-baselines3[extra]" gymnasium panda-gym huggingface_sb3 imageio imageio-ffmpeg moviepy

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "PandaReachDense-v3"
MODEL_NAME = f"a2c-{ENV_ID}"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

import imageio, numpy as np
import gymnasium as gym
import panda_gym
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

import base64
from IPython.display import HTML, display

def show_video(path, width=500):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 2. The environment
This time the observation is a **dict**, not a single array:
- `observation`: position and velocity of the end-effector (6 numbers)
- `desired_goal`: target position (3 numbers)
- `achieved_goal`: current end-effector position (3 numbers)

That's why the policy is `MultiInputPolicy` (`MlpPolicy` does not accept dict observations).

**Action:** 3 continuous values (movement along x, y, z) in [−1, 1] — not buttons!

**Reward (dense):** `−(distance between end-effector and target)` at every step. The closer to the target, the closer the reward is to 0.

In [ ]:
env = gym.make(ENV_ID)
obs, _ = env.reset(seed=0)
for k, v in obs.items():
    print(f"{k:14s} {v.round(3)}")
print("Action space:", env.action_space)
env.close()

## 3. Video helper
Plays a few episodes and records a video. `model=None` → random motion.

In [ ]:
def record(model=None, vecnorm=None, n_episodes=4, name="panda"):
    env = gym.make(ENV_ID, render_mode="rgb_array")
    frames, returns = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        total, done = 0.0, False
        while not done:
            if model is None:
                action = env.action_space.sample()
            else:
                o = {k: v[None] for k, v in obs.items()}
                if vecnorm is not None:
                    o = vecnorm.normalize_obs(o)
                action, _ = model.predict(o, deterministic=True)
                action = action[0]
            obs, r, terminated, truncated, _ = env.step(action)
            total += r
            frames.append(env.render())
            done = terminated or truncated
        returns.append(total)
    env.close()
    imageio.mimsave(f"{name}.mp4", frames, fps=25, macro_block_size=1)
    print("Episode returns:", np.round(returns, 2))
    show_video(f"{name}.mp4", width=480)

record(model=None, name="random_arm")

## 4. Normalisation and model
- `make_vec_env(n_envs=4)`: 4 environments in parallel.
- `VecNormalize`: rescales observations and rewards to **zero mean, unit variance**, which makes learning more stable. Its statistics (`vec_normalize.pkl`) must be saved with the model, otherwise the network sees differently scaled inputs at test time.
- `A2C("MultiInputPolicy")`: actor and critic are two heads of the same network.

In [ ]:
env = make_vec_env(ENV_ID, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

model = A2C(policy="MultiInputPolicy", env=env, device="cpu", verbose=0)

## 5. Training (~15–30 min)

In [ ]:
model.learn(total_timesteps=1_000_000, progress_bar=True)
model.save(MODEL_NAME)
env.save("vec_normalize.pkl")

## 6. Evaluate
The test environment uses `training=False` (freeze the statistics) and `norm_reward=False` (report the real reward).

In [ ]:
def make_eval_env():
    e = DummyVecEnv([lambda: Monitor(gym.make(ENV_ID, render_mode="rgb_array"))])
    e = VecNormalize.load("vec_normalize.pkl", e)
    e.training = False
    e.norm_reward = False
    return e

eval_env = make_eval_env()
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= -3.5 else 'NOT YET ❌'}")

## 7. Watch the trained arm

In [ ]:
record(model, vecnorm=eval_env, name="trained_arm")

## 8. Push to the Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_sb3 import package_to_hub

package_to_hub(
    model=model,
    model_name=MODEL_NAME,
    model_architecture="A2C",
    env_id=ENV_ID,
    eval_env=make_eval_env(),
    repo_id=REPO_ID,
    commit_message="Upload A2C PandaReachDense-v3 agent",
)

## 9. Check

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("PandaReachDense-v3", "stable-baselines3", -3.5)